In [1]:
!pip install pymysql

### 5.2.4. 파이마이에스큐엘로 버전 정보 확인하기

데이터베이스에서 변경된 내역을 영구적으로 확정하는 것을 커밋이라고 함.  
connection 객체의 autocommit 속성은 기본적으로 False이므로 INSERT, UPDATE, DELETE 문을 반영해 데이터를 변경하더라도 connection.commit()  함수를 호출해야 실제로 데이터베이스에 반영됨.

In [2]:
import pymysql

# connection 객체를 생성하여 연결을 진행함.
connection = pymysql.connect(host = 'localhost', port = 3306, db = 'INVESTAR'
                             ,user = 'root', passwd = '1234', autocommit = True)

# 커서 객체를 생성함.DB 커서는 Fetch 동작을 관리하는데 사용되는데, 만약 DB 자체가 커서를 지원하지 않으면, 
#Python DB API에서 이 커서 동작을 Emulation 하게 된다.
cursor = connection.cursor()

# 원하는 SQL문을 실행한다.
cursor.execute('select version();')

# 실행결과를 튜플로 받음
result = cursor.fetchone()

print(f'Maria DB version : {result}')

connection.close()

Maria DB version : ('10.8.3-MariaDB',)


## 5.3. 주식 시세를 매일 DB로 업데이트 하기

사실 필요할때마다 받아서 데이터분석을 진행하면 되긴 하지만, 이렇게 자동으로 쌓아서 얻는 이점이 있지 않을까 싶긴 함. 

### 5.3.1. DBUpdater 클래스 구조

### 5.3.2. 헤이디에스큐엘로 테이블 생성하기

In [ ]:
CREATE TABLE IF NOT EXISTS company_info (
    code VARCHAR(20)
    ,company VARCHAR(40)
    ,last_update DATE
    ,PRIMARY KEY (code)
);

CREATE TABLE IF NOT EXISTS daily_price (
    code VARCHAR(20)
    ,date DATE
    ,open BIGINT(20)
    ,high BIGINT(20)
    ,low BIGINT(20)
    ,close BIGINT(20)
    ,diff BIGINT(20)
    ,volumne BIGINT(20)
    ,PRIMARY KEY (code, date)
);

### 5.3.4. 파이마이에스큐엘로 테이블 생성하기

In [12]:
password = 1234
connection= pymysql.connect(host = 'localhost',user = 'root', port = 3306
                                   ,password = str(password), db = 'INVESTAR', charset = 'utf8')
cursor = connection.cursor()

#현재 어떤 테이블들이 있는지 확인함.
cursor.execute('show tables;');cursor.fetchall()

In [ ]:
import pymysql

class DBUpdater(): 
    
    def __init__(self, password) : 
#         self.password = str(password)
        self.conn = pymysql.connect(host = 'localhost',user = 'root', port = 3306
                                   ,password = str(password), db = 'INVESTAR', charset = 'utf8')
    
        #두 테이블을 생성해버린다. 
        with self.conn.cursor() as curs : 
            sql = """
            CREATE TABLE IF NOT EXISTS company_info (
                code VARCHAR(20)
                ,company VARCHAR(40)
                ,last_update DATE
                ,PRIMARY KEY (code)
            );
            """
            curs.execute(sql)

            sql = """
            CREATE TABLE IF NOT EXISTS daily_price (
                code VARCHAR(20)
                ,date DATE
                ,open BIGINT(20)
                ,high BIGINT(20)
                ,low BIGINT(20)
                ,close BIGINT(20)
                ,diff BIGINT(20)
                ,volumne BIGINT(20)
                ,PRIMARY KEY (code, date)
            );
            """
            curs.execute(sql)
        self.conn.commit()
        self.codes = dict()
#         self.update_comp_info() -> 아직 함수 선언 이전임
    
    def __del__(self) : 
        """소멸자 : MariaDB 연결 해제"""
        self.conn.close()

## 5.3.5. 종목코드 구하기 

이 부분부터는 추후 야파에서 긁어와서 업데이트 하는 걸로 바꿔야 함. 

In [ ]:
"https://kind.krx.co.kr/corpgeneral/corpList.do?method=download&pageIndex=1&currentPageSize=3000&comAbbrv=&beginIndex=&orderMode=3&orderStat=D&isurCd=&repIsuSrtCd=&searchCodeType=&marketType=&searchType=13&industry=&fiscalYearEnd=all&comAbbrvTmp=&location=all")